In [ ]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

# Configuración de rutas y parámetros
query_image_path = "../data/test_images/my_messy_500_peso_photo.jpg"
database_dir = "../data/database/sift_database/"
RATIO_THRESH = 0.7
MIN_MATCHES = 10

def recognize_banknote(query_path, db_dir):
    print(f"Analizando: {os.path.basename(query_path)}")
    
    # Carga de imagen y detección de características
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: No se pudo cargar la imagen.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    kp_query, des_query = sift.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No se encontraron descriptores en la imagen.")
        return

    matcher = cv2.BFMatcher(cv2.NORM_L2)
    category_votes = defaultdict(int)
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    # Comparación con la base de datos
    for db_file in db_files:
        des_db = np.load(db_file)
        filename = os.path.basename(db_file)
        category = filename.split("_comp_")[0].replace("norm_clean_", "")
        
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # Conteo de coincidencias válidas (Lowe's ratio test)
        good_matches = sum(1 for m_n in matches 
                           if len(m_n) == 2 and m_n[0].distance < RATIO_THRESH * m_n[1].distance)
                    
        if good_matches >= MIN_MATCHES:
            category_votes[category] += good_matches

    # Visualización de resultados
    if not category_votes:
        print("Resultado: Billete no reconocido.")
    else:
        results = sorted(category_votes.items(), key=lambda x: x[1], reverse=True)
        winner, score = results[0]
        
        print("-" * 40)
        print(f"BILLETE DETECTADO: {winner}")
        print(f"Puntos de coincidencia: {score}")
        print("-" * 40)
        
        if len(results) > 1:
            print(f"Candidato secundario: {results[1][0]} ({results[1][1]} matches)")

recognize_banknote(query_image_path, database_dir)

--- Analyzing Query Image: my_messy_500_peso_photo.jpg ---
  Matched 26 points with 100PesosFront component.
  Matched 72 points with 50PesosPolimeroFront component.
  Matched 23 points with 50PesosPolimeroBack component.
  Matched 83 points with 50PesosPolimeroBack component.
  Matched 38 points with 20PesosPolimeroBack component.
  Matched 23 points with 100PesosBack component.
  Matched 23 points with 20PesosBack component.
  Matched 36 points with 20PesosPolimeroFront component.
  Matched 17 points with 20PesosBack component.
  Matched 46 points with 50PesosPolimeroBack component.
  Matched 21 points with 20PesosPolimeroFront component.
  Matched 107 points with 500PesosBack component.
  Matched 44 points with 500PesosBack component.
  Matched 36 points with 100PesosBack component.
  Matched 32 points with 20PesosFront component.
  Matched 75 points with 1000PesosFront component.
  Matched 36 points with 200PesosFront component.
  Matched 23 points with 50PesosPolimeroFront compone